# Salient Object Detection — Demo Notebook

This notebook loads the trained improved model and runs it on uploaded images, showing:
- Input image
- Predicted saliency mask
- Overlay (red = salient region)
- Inference time per image

**Setup:** Make sure `sod_model.py` is in the same folder, and you have a checkpoint at `checkpoints/best_improved.pth`.

In [ ]:
# ===== 1. Imports & model loading =====
import time
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

from sod_model import SODNetImproved

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

CKPT_PATH = "checkpoints/best_improved.pth"   # adjust if needed

model = SODNetImproved().to(device)
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded model from epoch {ckpt['epoch']}")

In [ ]:
# ===== 2. Inference helper =====
def predict(image_path_or_array):
    """Run inference on a file path or a numpy RGB image.
    Returns (input_resized, mask, overlay, inference_ms)."""
    if isinstance(image_path_or_array, str):
        img = cv2.imread(image_path_or_array)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = image_path_or_array

    img_resized = cv2.resize(img, (128, 128))
    tensor = torch.from_numpy(img_resized.astype(np.float32) / 255.0)
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to(device)

    t0 = time.perf_counter()
    with torch.no_grad():
        pred = model(tensor)
    inference_ms = (time.perf_counter() - t0) * 1000

    mask = (pred[0, 0].cpu().numpy() > 0.5).astype(np.uint8) * 255
    red = np.zeros_like(img_resized)
    red[..., 0] = 255
    overlay = np.where(mask[..., None] > 0,
                       (0.5 * img_resized + 0.5 * red).astype(np.uint8),
                       img_resized)
    return img_resized, mask, overlay, inference_ms

In [ ]:
# ===== 3. Run on a sample image =====
# Replace with the path to any image on disk
SAMPLE = "data/ECSSD/test/image/0001.jpg"

img, mask, overlay, ms = predict(SAMPLE)
print(f"Inference: {ms:.1f} ms")

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(img);                ax[0].set_title("Input");           ax[0].axis("off")
ax[1].imshow(mask, cmap="gray");  ax[1].set_title("Predicted Mask"); ax[1].axis("off")
ax[2].imshow(overlay);            ax[2].set_title("Overlay");         ax[2].axis("off")
plt.show()

## Interactive Gradio app (optional)

Run the cell below to launch the Gradio web app inside the notebook.

In [ ]:
import gradio as gr

def predict_gradio(input_image):
    if input_image is None:
        return None, None, "Upload an image first."
    _, mask, overlay, ms = predict(input_image)
    info = f"Inference: {ms:.1f} ms  |  Image size: 128x128  |  Device: {device}"
    return mask, overlay, info

demo = gr.Interface(
    fn=predict_gradio,
    inputs=gr.Image(type="numpy", label="Input Image"),
    outputs=[
        gr.Image(label="Predicted Saliency Mask"),
        gr.Image(label="Overlay (red = salient region)"),
        gr.Textbox(label="Info"),
    ],
    title="Salient Object Detection - ECSSD",
    description="Upload an image. The CNN predicts which pixels belong to the most salient object.",
    allow_flagging="never",
)
demo.launch(share=True)